# Sesión 07 - Lab 1: Objetos de la capa Gold (Setup)

Este laboratorio quedó dividido en dos archivos por una limitación real de la plataforma. `CREATE MATERIALIZED VIEW` y `CREATE STREAMING TABLE` no corren sobre compute serverless de propósito general (el mismo donde corre el resto del curso): fallan con el error `MATERIALIZED_VIEW_OPERATION_NOT_ALLOWED.MV_NOT_ENABLED_ON_SERVERLESS_GENERIC_COMPUTE`, incluso usando una celda `%sql`, porque el error depende del compute al que está conectado el notebook, no del lenguaje de la celda. Necesitan un SQL Warehouse (Serverless o Pro), y un SQL Warehouse no tiene sesión de Spark, así que tampoco corre `spark.read.csv()` ni `dbutils`.

Por eso:
- **Este notebook (`sesion07_lab1.ipynb`)** corre en tu compute habitual (serverless de uso general o cluster) y se encarga de todo lo que necesita PySpark/`dbutils`: cargar las tablas Silver y dejar los archivos de la Streaming Table en el Volume.
- **`sesion07_lab1.sql`** corre conectado a un SQL Warehouse y contiene Lab 1A a Lab 1F: la View, la Materialized View, la Streaming Table y su comparación.

Datos de este laboratorio:
- `tickets_gold_inicial.csv`: 35 tickets de soporte ya en forma Silver (resultado de un pipeline de limpieza como el de las Sesiones 05-06), con algunas filas con problemas de calidad a propósito (el Lab 2 los va a validar).
- `tickets_stream_lote1.csv` / `tickets_stream_lote2.csv`: dos lotes de archivos para la Streaming Table, simulando llegada progresiva.
- `sla_prioridad.csv`: tabla de referencia chica (horas de SLA y departamento responsable por prioridad).

## Verificación del entorno

In [ ]:
dbutils.fs.ls("/Volumes/dbassociate/default/vol_landing/sesion_07")

## Setup: aterrizar la tabla Silver de origen

Este laboratorio no repite la limpieza de Bronze a Silver (ya se practicó a fondo en las Sesiones 05 y 06): el foco de hoy es lo que viene después, así que el CSV ya llega en forma Silver y se aterriza directo.

In [ ]:
df_tickets = spark.read.csv(
    "/Volumes/dbassociate/default/vol_landing/sesion_07/tickets_gold_inicial.csv",
    header=True,
    inferSchema=True,
)
df_tickets.write.mode("overwrite").saveAsTable("dbassociate.silver.tickets_soporte")

df_sla = spark.read.csv(
    "/Volumes/dbassociate/default/vol_landing/sesion_07/sla_prioridad.csv",
    header=True,
    inferSchema=True,
)
df_sla.write.mode("overwrite").saveAsTable("dbassociate.silver.sla_prioridad")

print("Tickets cargados:", spark.table("dbassociate.silver.tickets_soporte").count())

## Preparar el origen de la Streaming Table: primer lote

`sesion07_lab1.sql` (Lab 1E) va a crear la Streaming Table apuntando a esta carpeta del Volume. Subimos acá el primer lote antes de pasar al archivo SQL.

In [ ]:
dbutils.fs.mkdirs("/Volumes/dbassociate/default/vol_landing/sesion_07/tickets_stream")
dbutils.fs.cp(
    "/Volumes/dbassociate/default/vol_landing/sesion_07/tickets_stream_lote1.csv",
    "/Volumes/dbassociate/default/vol_landing/sesion_07/tickets_stream/tickets_stream_lote1.csv",
)

print("Primer lote listo en el Volume.")

**Seguí en `sesion07_lab1.sql`**, conectado a un SQL Warehouse (Serverless o Pro): ahí están Lab 1A a Lab 1F. Volvé a este notebook solo cuando ese archivo te pida aterrizar el segundo lote (el paso de abajo).

## Paso intermedio: aterrizar el segundo lote de la Streaming Table

Corré esta celda cuando `sesion07_lab1.sql` llegue a la sección de `REFRESH STREAMING TABLE` con el segundo lote (después de Lab 1E).

In [ ]:
dbutils.fs.cp(
    "/Volumes/dbassociate/default/vol_landing/sesion_07/tickets_stream_lote2.csv",
    "/Volumes/dbassociate/default/vol_landing/sesion_07/tickets_stream/tickets_stream_lote2.csv",
)

print("Segundo lote listo en el Volume. Volvé a sesion07_lab1.sql para refrescar la Streaming Table.")

## Limpieza

In [ ]:
spark.sql("DROP TABLE IF EXISTS dbassociate.silver.tickets_soporte")
spark.sql("DROP TABLE IF EXISTS dbassociate.silver.sla_prioridad")

# Los objetos Gold (view, materialized view, streaming table) se eliminan
# desde la celda de Limpieza de sesion07_lab1.sql, no desde acá.

print("Tablas Silver de este laboratorio eliminadas.")